## Compile modeling dataframe

This notebook is a customized project for separate pre-processing of indicators, covariates and spectra from the EU funded project [AI4SoilHealth](https://ai4soilhealth.eu). The project requires customization of the xspatula core framework. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". 

The Machine Learning (ML) modeling as structured in the xspatula_ai4sh framework requires that all model data is compiled in a separate framework; this notebook compiles soil indicators, spectra and covariates for use in the ML modeling.

## How to use this notebook

This notebook allows you to compile different dataframes, and apply pre-processing to both indicators and spectra. You need to compile different dataframes for different regressors, and you can also compile dataframes to compare the effects of different pre-processing chains. Cell 4.1.1 illustrates how the compilation and pre-processing can be integrated in a single JSON process file.

For detailed instructions on this notebook, please consult the online manual for [Compile Dataframe](https://xspatula.github.io/xspatula_ai4sh_docs/machine_learning/compile).

In [ ]:
# Standard library imports
from os import path, getcwd

import sys 

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:
    
    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file  

scheme_file = '../scheme_ai4sh.json'

# 1 Compile dataframes
Dataframes for modeling must contain the indicator (target feature) that the modeling aims to predict. It must also contain at least one covariate. In the examples below, spectra are always included (spectra are a special class of covariates) and in some dataframes also soil texture is included as an additional covariate.

All compiled dataframes will be saved to your local machine.

## 1.1 Compile dataframes with indicators and spectra

Indicators can be compiled either as raw data, or after pre-processing. Spectra must be compiled in its raw (non pre-processed) format. After compilation you can apply a pre-existing pre-processing chain also for the spectra.

### 1.1.1 Compile raw dataframe with indicators and spectra

Cell 1.1.1 illustrates how to compile raw indicators and spectra.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/compile_neospectra_10nm_raw.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

## 1.1.2 Compile dataframe with pre-processed indicators

You can also compile a dataframe with pre-processed indicators, but not with pre-processed spectra. For the spectral pre-processing to be correct it must be done against a pre-existing pipeline chain in the compiled dataframe to avoid overfitting - that is done in cell 3.1.1.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/compile_neospectra_10nm_olif.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 1.1.3 Pre-process indicators in the compiled dataframe
You can apply indicator pre-processing in a compiled dataframe. Cell 1.1.3 applies the `indicator_range` process to already compiled data - see the JSON process file for details. The compiled dataframe is saved as `neospectra_10nm.ir` in the same folder as the compiled source data.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/indicator_pre-process_compiled_neospectra_10nm.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2 Compile dataframe with covariates

Covariates are selected separately (process `select_covariate_only`) and then compiled with the indicators and spectra. There is no notebook for selecting covariates in the default setup and you thus have to run the process `select_covariate_only` first (cell 2.1.1) and then compile the dataframe with covariates included (cell 2.1.2)

## 2.1 Select covariates

The AI4SH dataset contains textural covariates, but these were not selected and written to a dataframe yet, that is done in cell 2.1.1.


### 2.1.1 Select soil textural classes as covariates

The cell below selects the USDA soil texture class and the USDA texture major group (a predefined grouping of the classes) for the AI4SH samples; the next cell (2.1.2) compiles them with the indicators and the spectra.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/select/select_ai4sh_texture.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.2 Compile dataframe with raw indicators, spectra and covariates

Compile a new dataframe that includes raw indicators, spectra and covariates. The companion JSON of the compiled dataset lists which columns are indicators, covariates and spectra (`column_blocks`).

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/compile_neospectra_10nm_texture_raw.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.3 Compile dataframe with pre-processed indicators, and raw spectra and covariates

Compile a new dataframe that includes pre-processed indicators, and raw spectra and raw covariates.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/compile_neospectra_10nm_texture_olif.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 3 Pre-process spectra in compiled dataframes

Spectra need to be added to a compiled dataframe as raw spectra and then you can call a pre-processing chain to apply any pre-processing chain that you saved and kept.

## 3.1 Pre-process spectra from saved pipeline chain
You apply pre-processing to spectra by calling an existing (saved) dataframe (or rather its accompanying JSON file) with a preset pre-processing chain for invoking a pre-processing pipeline. The examples in cells 3.1.1-3.1.3 all call the same pre-processing chain, defined in the dataframe `1350-2500_10.snv.d1.mc.pca5.json`. You can call dataframes from other projects, or stored in a dedicated folder anywhere, to invoke a pre-processing chain, but a prerequisite is that the spectral input data has exactly the same range, resolution and units as the dataframe you want to run the chain for.

### 3.1.1 Pre-process the spectra in dataframe with raw indicators

Cell 3.1.1 pre-processes the spectra in the compiled dataframe created in cell 1.1.1.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/pre-process_simple-chain_neospectra_10nm_raw.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.1.2 Pre-process the spectra in dataframe with pre-processed indicators

Cell 3.1.2 pre-processes the spectra in the compiled dataframe created in cell 1.1.2.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/pre-process_simple-chain_neospectra_10nm_olif.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.1.3 Pre-process the spectra in dataframe with pre-processed indicators and covariates

Cell 3.1.3 pre-processes the spectra in the compiled dataframe created in cell 2.1.3.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/pre-process_simple-chain_neospectra_10nm_olif_texture.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.1.4 Pre-process the spectra with SNV, derivative and mean centring, without PCA

The spectral chain for PLSR and the other multivariate regressors in `ml_model.ipynb` (cells 2.2.1–2.2.5): the same chain as 3.1.3 but without the final PCA, so the regressors see all 116 bands. PLSR makes its own components; after a `pca5` it can not use more than 5 (plus the covariate columns). Answer `y` to save the result `neospectra_10nm_olif_texture.snv.d1.mc`.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/pre-process_snv-d1-mc_neospectra_10nm_olif_texture.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 4 Select, compile and pre-process in a single process file
Instead of stepwise cells for separate selection of indicators and spectra (and covariates), followed by compiling and then invoking a spectral pre-processing on the compiled spectra in 4 to 5 separate cells and process files, you can do it from a single cell and with a single process file. A prerequisite is that the spectral data pre-process pipeline already exists.

## 4.1 Replace 4 cells and 4 process files with singles 
A JSON process file can stack any number of processes that are then executed sequentially. The JSON process file called from cell 4.1.1 stacks two processes, `compile_dataset_from_db` and `chemometric_chain` in a single process file. The process `compile_dataset_from_db` includes indicator data range cutting on the fly. The spectra pre-process chain called from `chemometric_chain` process must exist beforehand.

### 4.1.1 Integrating dataframe compilation directly from the database with indicator and spectra pre-processing

Cell 4.1.1 replicates the processes from four (4) previous cells in the notebook `ml_preprocess_separated` and this notebook. The output is a dataframe with pre-processing applied both to the indicators and the spectra.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/compile/compile_wetlab+neospectra_10nm_from_db.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 5 Inspect dataframe

You can inspect any dataframe after it has been created.

## 5.1 Inspect compiled dataframe

Compiled dataframes can be very large and you will probably want to specify what you want to see. See the documentation page [Parquet files - inspect the dataset](https://xspatula.github.io/xspatula_ai4sh_docs/machine_learning/compile/#inspect-the-compiled-dataframe).

### 5.1.1 Define dataframe and columns and rows to inspect

The default setting is to inspect the most recently created dataframe. Change to a specific dataframe to inspect in the JSON process file. You can also specify which columns, and how many rows, to inspect.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/inspect/inspect_latest_compiled_df.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)